In [3]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

import torchvision
import torchvision.transforms as transforms

In [4]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

Device: cpu


In [5]:
CONFIG = {
    "mnist": {
        "batch_size": 128,
        "epochs": 10,
    },

    "cifar10": {
        "batch_size": 128,
        "epochs": 30,
    }
}

In [6]:
from pathlib import Path

ROOT = Path.cwd()

# If the notebook is running from the notebooks/ directory
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT)
print("Data directory:", DATA_DIR)

Project root: /home/mahdi/Documents/cross-norm-adversarial-robustness
Data directory: /home/mahdi/Documents/cross-norm-adversarial-robustness/data


In [7]:
mnist_train_transform = transforms.Compose([
    transforms.ToTensor(),
])

mnist_test_transform = transforms.Compose([
    transforms.ToTensor(),
])


cifar_train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
])

cifar_test_transform = transforms.Compose([
    transforms.ToTensor(),
])

In [8]:
mnist_train_dataset = torchvision.datasets.MNIST(
    root=DATA_DIR,
    train=True,
    download=True,
    transform=mnist_train_transform,
)

mnist_test_dataset = torchvision.datasets.MNIST(
    root=DATA_DIR,
    train=False,
    download=True,
    transform=mnist_test_transform,
)


cifar_train_dataset = torchvision.datasets.CIFAR10(
    root=DATA_DIR,
    train=True,
    download=True,
    transform=cifar_train_transform,
)

cifar_test_dataset = torchvision.datasets.CIFAR10(
    root=DATA_DIR,
    train=False,
    download=True,
    transform=cifar_test_transform,
)

100.0%
100.0%
100.0%
100.0%
100.0%


In [9]:
print("MNIST train:", len(mnist_train_dataset))
print("MNIST test :", len(mnist_test_dataset))

print("CIFAR-10 train:", len(cifar_train_dataset))
print("CIFAR-10 test :", len(cifar_test_dataset))

MNIST train: 60000
MNIST test : 10000
CIFAR-10 train: 50000
CIFAR-10 test : 10000


In [10]:
mnist_train_loader = DataLoader(
    mnist_train_dataset,
    batch_size=CONFIG["mnist"]["batch_size"],
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

mnist_test_loader = DataLoader(
    mnist_test_dataset,
    batch_size=CONFIG["mnist"]["batch_size"],
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)


cifar_train_loader = DataLoader(
    cifar_train_dataset,
    batch_size=CONFIG["cifar10"]["batch_size"],
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

cifar_test_loader = DataLoader(
    cifar_test_dataset,
    batch_size=CONFIG["cifar10"]["batch_size"],
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

In [14]:
mnist_images, mnist_labels = next(iter(mnist_train_loader))
cifar_images, cifar_labels = next(iter(cifar_train_loader))

print("MNIST")
print("Images shape:", mnist_images.shape)
print("Labels shape:", mnist_labels.shape)
print("Pixel range:", mnist_images.min().item(), mnist_images.max().item())

print()

print("CIFAR-10")
print("Images shape:", cifar_images.shape)
print("Labels shape:", cifar_labels.shape)
print("Pixel range:", cifar_images.min().item(), cifar_images.max().item())

MNIST
Images shape: torch.Size([128, 1, 28, 28])
Labels shape: torch.Size([128])
Pixel range: 0.0 1.0

CIFAR-10
Images shape: torch.Size([128, 3, 32, 32])
Labels shape: torch.Size([128])
Pixel range: 0.0 1.0


In [15]:
class MNISTCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.MaxPool2d(kernel_size=2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 14 * 14, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [16]:
mnist_model = MNISTCNN().to(device)

print(mnist_model)

MNISTCNN(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU()
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=12544, out_features=128, bias=True)
    (2): ReLU()
    (3): Linear(in_features=128, out_features=10, bias=True)
  )
)


In [17]:
dummy_input = torch.randn(8, 1, 28, 28).to(device)

dummy_output = mnist_model(dummy_input)

print("Input shape :", dummy_input.shape)
print("Output shape:", dummy_output.shape)

Input shape : torch.Size([8, 1, 28, 28])
Output shape: torch.Size([8, 10])
